In [0]:
%pip install git+https://github.com/databrickslabs/lakebridge
dbutils.library.restartPython()

In [0]:
%sql
use catalog learn_adb_fikrat;
create schema if not exists aw

In [0]:
%sql
use aw;
CREATE TABLE Address(
	AddressID int ,
	AddressLine1 string ,
	AddressLine2 string,
	City string,
	StateProvince string,
	CountryRegion string,
	PostalCode string,
	rowguid string,
	ModifiedDate timestamp )


In [0]:
from databricks.sdk import WorkspaceClient

from databricks.labs.lakebridge.config import (
    DatabaseConfig,
    ReconcileConfig,
    ReconcileMetadataConfig,
    TableRecon
)
from databricks.labs.lakebridge.reconcile.recon_config import (
    Table,
    ColumnMapping,
    ColumnThresholds,
    Transformation,
    JdbcReaderOptions,
    Aggregate,
    Filters
)
from databricks.labs.lakebridge.reconcile.execute import (
    recon,
    reconcile_aggregates
)
from databricks.labs.lakebridge.reconcile.exception import ReconciliationException


In [0]:
from databricks.labs.lakebridge.config import (
    DatabaseConfig,
    ReconcileConfig,
    ReconcileMetadataConfig
)

reconcile_config = ReconcileConfig(
         data_source = "tsql",
         report_type = "all",
         secret_scope = "remorph_tsql",
         database_config= DatabaseConfig(source_catalog="AdventureWorks",
                                         source_schema="SalesLT",
                                         target_catalog="learn_adb_fikrat",
                                         target_schema="aw"
                          ),
         metadata_config = ReconcileMetadataConfig(
              catalog = "learn_adb_fikrat",
              schema= "reconcile"
         )
    )

In [0]:
from databricks.labs.lakebridge.config import TableRecon
from databricks.labs.lakebridge.reconcile.recon_config import (
    Table,
    ColumnMapping,
    ColumnThresholds,
    TableThresholds,
    Transformation,
    JdbcReaderOptions,
    Aggregate,
    Filters
)

table_recon = TableRecon(
    source_schema="SalesLT",
    target_catalog="learn_adb_fikrat",
    target_schema="aw",
    tables=[
        Table(
            source_name="Address",
            target_name="Address",
            join_columns= ["AddressID"]) 
    ]
)

In [0]:
from databricks.labs.lakebridge import __version__
from databricks.sdk import WorkspaceClient

from databricks.labs.lakebridge.reconcile.trigger_recon_service import TriggerReconService
from databricks.labs.lakebridge.reconcile.exception import ReconciliationException

ws = WorkspaceClient(product="lakebridge", product_version=__version__)


try:
  result = TriggerReconService.trigger_recon(
            ws = ws,
            spark = spark, # notebook spark session
            table_recon = table_recon, # previously created
            reconcile_config = reconcile_config # previously created
          )
  print(result.recon_id)
  print(result)
  print("***************************")
except ReconciliationException as e:
    recon_id = e.reconcile_output.recon_id
    print(f" Failed : {recon_id}")
    print(e)
    print("***************************")
except Exception as e:
    print(e.with_traceback)
    raise e
    print(f"Exception : {str(e)}")
    print("***************************")


In [0]:
%sql
select * from aw.address

In [0]:
%sql 
select * from learn_adb_fikrat.reconcile.details

In [0]:
from databricks.labs.lakebridge.transpiler.execute import (transpile,transpile_sql)      

from databricks.labs.lakebridge.transpiler.execute import (
    Table,
    ColumnMapping,
    ColumnThresholds)

In [0]:
from databricks.labs.lakebridge.transpiler.transpile_engine.transpile_config import (transpile_config
)


In [0]:
from databricks.labs.lakebridge.transpiler.execute import (
    transpile,
    transpile_sql)

# Example usage of transpile function
# transpiled_code = transpile(
    
#     source_type="sql",
#     source_code="SELECT * FROM SalesLT.Address",
#     target_type="spark"
# )

# print(transpiled_code)

# Example usage of transpile_sql function
transpiled_sql = transpile_sql(
    source_sql="SELECT * FROM SalesLT.Address"
    
)

print(transpiled_sql)